# Etape 2 : Questions-Réponses

## Introduction

Dans ce notebook je mets en place deux approches pour trouver les réponses les plus proches sémentiquement paralnt de la réponse initiale.

- **Word2Vec**
Word2Vec permet de creer pour chaque mot un vecteur selon le context donné par les corpus. Deux mots proches sementiquement auront donc des vecteurs "similaires". Cependant pour les réponses nous travaillons sur de phrases, j'ai donc choisis que chaque phrase serait vectoriser par un vercteur moyen des vecteurs des mots qui la compose. Cette approche me semblant peu rigoureuse il faudrait encore identifier ses limitations et les incoherences pouvant en résulter.

- **Doc2Vec**
Doc2Vec est davantage coherent puisqu'il cree directement des vecteur pour des paragraphe et non des mots. J'ai donc choisi de l'utiliser egalement afin de pouvoir comparer les deux approches.

In [8]:
#imports
import pandas as pd
from gensim.models import Word2Vec
from gensim.models import Doc2Vec
import gensim.utils
import gensim.models.doc2vec

In [9]:
#data
data = pd.read_csv('../../data/train.csv') 
data = data.dropna(subset=['Context','Response'])
data_unique = data.drop_duplicates(subset=['Context','Response']) 
test_data = data_unique.sample(n=10,random_state=42) 
train_data = data_unique.drop(test_data.index) 
train_data.to_csv('../../data/train_unique_e2p2.csv',index=False) 
test_data.to_csv('../../data/test_unique_e2p2.csv',index=False)
test_data.head()



,Context,Response
3201,What makes a healthy marriage last?,This answer varies based on you relationship. ...
3365,I have a friend that who I used to be in a rel...,"It is not the case of being right or wrong, in..."
1359,My mother takes care of niece whom my sister a...,This sounds like a possible boundary issue. Bo...
1702,I am a heterosexual male in my late 20s. I fin...,If you enjoy cross-dressing and are comfortabl...
3397,Or how to send him somewhere that can help him...,Your dad needs to be aware that he has a probl...


### Fonctions utilitaires

In [113]:
def read_corpus_from_df(df, tokens_only=False):
    for i, text in enumerate(df['Response']):
        tokens = gensim.utils.simple_preprocess(str(text))
        if tokens_only:
            yield tokens
        else:
            yield gensim.models.doc2vec.TaggedDocument(tokens, [i])

In [114]:
import textwrap

def print_wrapped(label, text, width=100):
    print(f"{label}")
    print(textwrap.fill(text, width=width))
    print()

## Approche Word2Vec

### Références

- [Gensim Word2Vec documentation](https://radimrehurek.com/gensim/models/word2vec.html)


### Principe

Word2Vec permet d'entraine notre model sur les tokens du corpus.

Pour chaque phrase  **phrase** on fait alors la **moyenne** des vecteurs de ses mots afin d'obtenir son vecteur moyen.

On compare ensuite la question à toutes les réponse grace la **similarité cosinus**



### Tokenisation pour Word2Vec

In [ ]:
from gensim.models import Word2Vec
import gensim.utils

train_tokens_w2v = [
    gensim.utils.simple_preprocess(str(text))
    for text in pd.concat([train_data['Response'], train_data['Context']])

]

print(f"Nombre de réponses tokenizd : {len(train_tokens_w2v)}")
print(f"Exemple de tokens : {train_tokens_w2v[0][:10]}")

Nombre de réponses tokenizd : 5476
Exemple de tokens : ['if', 'everyone', 'thinks', 'you', 're', 'worthless', 'then', 'maybe', 'you', 'need']


### Entraînement du modèle Word2Vec

- **vector_size** : dimensions du vecteur
- **min_count** : nombre d'occurence minimale à partir du quel un mot est pris en compte
- **epochs** : nombre d'itération sur le corpus
- **workers** : permet de paralleliser l'entrainement
- **window** : permet d'elargir le contexte pris en compte au tour du mot

In [107]:
w2v_model = Word2Vec(sentences=train_tokens_w2v, vector_size=80, min_count=1,epochs=80, workers=4,window=10)

#Exemple
print(f"Le mot 'depression' apparait {w2v_model.wv.get_vecattr('depression', 'count')} fois dans le corpus d'entraînement.")
print(f"Représentation sous forme de vecteur à 80 dimensions du mot 'depression':\n {w2v_model.wv['depression']}")

Le mot 'depression' apparait 650 fois dans le corpus d'entraînement.
Représentation sous forme de vecteur à 80 dimensions du mot 'depression':
 [ 2.4410093   3.0558422   2.8419013   1.5169537   0.8714178  -1.0958709
  5.135343    2.8215144   2.5784392  -0.99773145 -1.706841   -3.5225573
 -0.37768748  3.2141092   2.1658792   3.2560964   2.7340407  -2.9751594
  6.496286    3.66513    -2.5508978  -0.630757    2.4182804   0.14066236
 -4.7085276  -1.6646423  -2.9499857  -1.1437771  -4.2335396   4.587395
 -1.6080847   3.4557447  -0.39144284  2.7825923   1.0556908  -1.3740636
  0.8194709  -3.8603182   0.5379346   3.4591055   4.6595173   0.4014386
 -0.8302024   1.986507   -6.1608324  -5.54801    -0.24528931 -2.3466082
  4.496157    2.0706654   1.5753087  -0.587998    4.379094   -2.4450734
  1.034786   -1.6759031  -4.5479536   0.82076794  3.0619822  -1.1983805
  0.31212214  2.5380926  -4.3207808   3.5475733   0.43158096  1.6899431
  2.7155602  -2.9957824   1.6815702   0.49644795 -1.5096947  -0.

### Vectorisation des réponses (moyenne des vecteurs de mots)

In [108]:
import numpy as np

def sentence_vector(tokens, model):
    vecs = [model.wv[word] for word in tokens if word in model.wv]
    if len(vecs) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

responses_train_w2v = train_data['Response'].reset_index(drop=True)
response_vectors_w2v = np.array([
    sentence_vector(gensim.utils.simple_preprocess(str(r)), w2v_model)
    for r in responses_train_w2v
])

print(f"Matrice de vecteurs de réponses : {response_vectors_w2v.shape}")

Matrice de vecteurs de réponses : (2738, 80)


### Fonction de recherche des K réponses les plus proches

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def k_response_w2v(question, k):
    #Vectoriser la question
    q_tokens = gensim.utils.simple_preprocess(str(question))
    q_vec = sentence_vector(q_tokens, w2v_model).reshape(1, -1)

    # mesure de la similarité avec toutes les réponse

    similarities = cosine_similarity(q_vec, response_vectors_w2v).flatten()

    #recup les k meilleurs indices
    top_k_indices = similarities.argsort()[-k:][::-1]

    return responses_train_w2v.iloc[top_k_indices].tolist(), similarities[top_k_indices]

### Test du modèle sur le jeu de test

In [110]:
for doc_id in range(len(test_data)):
    question = test_data.iloc[doc_id]['Context']
    true_response = test_data.iloc[doc_id]['Response']

    top_k_responses, top_k_scores = k_response_w2v(question, k=3)

    print("=== DOCUMENT DE TEST ===\n")
    print_wrapped('Context:', question)
    print_wrapped('Réponse initiale :', true_response)

    print(u'\n=== TOP 3 RÉPONSES SUGGÉRÉES PAR LE MODÈLE ===\n')
    for i, (resp, score) in enumerate(zip(top_k_responses, top_k_scores)):
        print(f"--- TOP {i+1} (score: {score:.4f}) ----")
        print_wrapped('', resp)

    print("\n" + "="*80 + "\n")

=== DOCUMENT DE TEST ===

Context:
What makes a healthy marriage last?

Réponse initiale :
This answer varies based on you relationship. However, I do believe their are some basic fundamental
areas that are beneficial for a healthy marriage:1.) Effective Communication2.) Trust3.)
Love/Passion4.) Loyalty. 5.) Unconditional Positive Regard. Everyone has their favorite qualities
they feel best fit a marriage. However, these are what I think are great starting points. 


=== TOP 3 RÉPONSES SUGGÉRÉES PAR LE MODÈLE ===

--- TOP 1 (score: 0.6032) ----

I get it. Your husband tells you that he's not in love with you, but oops, he's changed his mind and
will tolerate you for a while longer? Excuse me? My Dear, it's okay if you expect more than that
from a marriage. Maybe the question has shifted from whether he is happy in the marriage to whether
you are happy in the marriage. You say you love this man,  who makes you "feel like nothing". I say
it might be time to sit down with an individual th

## Approche Doc2Vec

### Références

- [Gensim Doc2Vec example](https://radimrehurek.com/gensim/auto_examples/tutorials/run_doc2vec_lee.html)
- [Gensim Doc2Vec documentation](https://radimrehurek.com/gensim/models/doc2vec.html)


### Tokenisation pour Doc2Vec

In [126]:
train_corpus = list(read_corpus_from_df(train_data))
test_corpus  = list(read_corpus_from_df(test_data, tokens_only=True))
print(train_corpus[:2])

[TaggedDocument(words=['if', 'everyone', 'thinks', 'you', 're', 'worthless', 'then', 'maybe', 'you', 'need', 'to', 'find', 'new', 'people', 'to', 'hang', 'out', 'with', 'seriously', 'the', 'social', 'context', 'in', 'which', 'person', 'lives', 'is', 'big', 'influence', 'in', 'self', 'esteem', 'otherwise', 'you', 'can', 'go', 'round', 'and', 'round', 'trying', 'to', 'understand', 'why', 'you', 're', 'not', 'worthless', 'then', 'go', 'back', 'to', 'the', 'same', 'crowd', 'and', 'be', 'knocked', 'down', 'again', 'there', 'are', 'many', 'inspirational', 'messages', 'you', 'can', 'find', 'in', 'social', 'media', 'maybe', 'read', 'some', 'of', 'the', 'ones', 'which', 'state', 'that', 'no', 'person', 'is', 'worthless', 'and', 'that', 'everyone', 'has', 'good', 'purpose', 'to', 'their', 'life', 'also', 'since', 'our', 'culture', 'is', 'so', 'saturated', 'with', 'the', 'belief', 'that', 'if', 'someone', 'doesn', 'feel', 'good', 'about', 'themselves', 'that', 'this', 'is', 'somehow', 'terrible',

### Entrainement du model

- **vector_size** : dimensions du vecteur
- **min_count** : nombre d'occurence minimale à partir du quel un mot est pris en compte
- **epochs** : nombre d'itération sur le corpus
- **workers** : permet de paralleliser l'entrainement
- **dm** : algo utilisé (j'ai pris celui par default pour l'instant)
- **window** : permet d'elargir le contexte pris en compte au tour du mot

- **total_example** : nombre de phrases

In [140]:
model = gensim.models.doc2vec.Doc2Vec(vector_size=50, min_count=1, epochs=40, workers=4, window=5)
model.build_vocab(train_corpus)
model.train(train_corpus, total_examples=model.corpus_count, epochs=model.epochs)

In [141]:
#Exemple
print(f"Le mot 'depression' apparait {model.wv.get_vecattr('depression', 'count')} fois dans le corpus d'entraînement.")
print(f"Représentation sous forme de vecteur à 50 dimensions du mot 'depression':\n {model.wv['depression']}")

Le mot 'depression' apparait 420 fois dans le corpus d'entraînement.
Représentation sous forme de vecteur à 50 dimensions du mot 'depression':
 [ 2.1163912   0.66409516  1.5766989  -1.9987563  -2.6176512  -1.067253
 -3.419838   -1.0377629   0.8136437  -1.3775498   3.0595357   1.5556283
 -0.07536417 -1.282272   -1.2824166  -2.0332053   0.6152848  -0.87982434
  1.5189899   1.2501822   1.6097904   0.6750264   0.7977067  -1.1652843
 -2.7220812   4.1659164  -1.0833049  -1.745964   -0.32153645 -1.8132523
  1.1547008  -3.215216    0.29193065  0.40461725 -0.13639842 -0.8945464
 -0.22535111 -1.7201201   1.7391104  -2.1845517   1.792893   -0.8034248
  1.1043509  -0.1281837  -0.4923994   0.61905444 -0.8097589   1.1550478
  0.7721092   0.0727075 ]


### Verification du model par inférence sur les données d'entrainement

In [142]:
ranks = []
second_ranks = []
for doc_id in range(len(train_corpus)):
    inferred_vector = model.infer_vector(train_corpus[doc_id].words)
    sims = model.dv.most_similar([inferred_vector], topn=len(model.dv))
    rank = [docid for docid, sim in sims].index(doc_id)
    ranks.append(rank)

    second_ranks.append(sims[1])

In [143]:
import collections

counter = collections.Counter(ranks)
print(counter)

Counter({0: 2028, 1: 708, 2270: 1, 2: 1})


### Test du model sur le jeu de test

In [144]:
# Pick a random document from the test corpus and infer a vector from the model
for doc_id in range(len(test_corpus)):
    q_tokens = gensim.utils.simple_preprocess(str(test_data.iloc[doc_id]['Context']))
    inferred_vector = model.infer_vector(q_tokens, epochs=100)
    sims = model.dv.most_similar([inferred_vector], topn=len(model.dv))

    # Context et réponse initiale
    print("=== DOCUMENT DE TEST ===\n")
    print("Context : ", ' '.join(test_data.iloc[doc_id]['Context'].split()))
    print_wrapped('Réponse initiale :', ' '.join(test_corpus[doc_id]))

    # 3 meilleures réponses suggérées (on skip la 1ère car c'est la réponse elle-même)
    print(u'\n=== TOP 3 RÉPONSES SUGGÉRÉES PAR LE MODÈLE ===\n')
    for i in range(3):
        print(f"--- TOP {i+1} (score: {sims[i][1]:.4f}) ---")
        print_wrapped('', ' '.join(train_corpus[sims[i][0]].words))


=== DOCUMENT DE TEST ===

Context :  What makes a healthy marriage last?
Réponse initiale :
this answer varies based on you relationship however do believe their are some basic fundamental
areas that are beneficial for healthy marriage effective communication trust love passion loyalty
unconditional positive regard everyone has their favorite qualities they feel best fit marriage
however these are what think are great starting points


=== TOP 3 RÉPONSES SUGGÉRÉES PAR LE MODÈLE ===

--- TOP 1 (score: 0.7011) ---

hard to say whole lot without knowing more however if you focus your attention on her what she
saying what she feeling instead of trying to make yourself heard and understood first that often
good step also work on building win win agreements with her and follow through on them those are the
areas see men fall short on most often hope that helps

--- TOP 2 (score: 0.6627) ---

it sounds like you ve already learned that just being honest is often the best approach what do you
t

## Approche BERT

*******\*TODO\********


## Conclusion

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# similirité moey w2v
w2v_scores = []
for text in test_data['Context']:
    q_tokens = gensim.utils.simple_preprocess(str(text))
    q_vec = sentence_vector(q_tokens, w2v_model).reshape(1, -1)
    # On prend le score du Top 1
    sims = cosine_similarity(q_vec, response_vectors_w2v).flatten()
    w2v_scores.append(np.max(sims))

#sims moey d2v
d2v_scores = []
for text in test_data['Context']:
    q_tokens = gensim.utils.simple_preprocess(str(text))
    inferred_vector = model.infer_vector(q_tokens)
    #score le + proche
    sims = model.dv.most_similar([inferred_vector], topn=1)
    d2v_scores.append(sims[0][1])

#tableau recapitulatif
comparison_results = pd.DataFrame({
    'Métrique': ['Moyenne Top 1', 'Score Max', 'Score Min'],
    'Word2Vec': [np.mean(w2v_scores), np.max(w2v_scores), np.min(w2v_scores)],
    'Doc2Vec': [np.mean(d2v_scores), np.max(d2v_scores), np.min(d2v_scores)]
})

print("Synthèse automatique des performances (Similarité Cosinus) :")
comparison_results.set_index('Métrique')

Synthèse automatique des performances (Similarité Cosinus) :


,Word2Vec,Doc2Vec
Métrique,,
Moyenne Top 1,0.829368,0.612254
Score Max,0.958324,0.755509
Score Min,0.603182,0.539677


In [148]:
# Extraction des moyennes pour la décision
avg_w2v = comparison_results.loc[0, 'Word2Vec']
avg_d2v = comparison_results.loc[0, 'Doc2Vec']
winner = "Word2Vec" if avg_w2v > avg_d2v else "Doc2Vec"

print(f"=== ANALYSE DE COHÉRENCE ===")
print(f"Model Word2Vec (vector_size={w2v_model.vector_size}, epochs={w2v_model.epochs}) : {avg_w2v:.4f}")
print(f"Model Doc2Vec (vector_size={model.vector_size}, epochs={model.epochs}) : {avg_d2v:.4f}")
print(f"\nConclusion : Le model **{winner}** est  le plus performant.")


=== ANALYSE DE COHÉRENCE ===
Model Word2Vec (vector_size=80, epochs=80) : 0.8294
Model Doc2Vec (vector_size=50, epochs=40) : 0.6123

Conclusion : Le model **Word2Vec** est  le plus performant.


### Word2Vec

Les resultats de Word2Vec sont assez surprenemment coherents... 

**TODO : meilleur interpretation des resultats**

### Doc2Vec

resultats peu insatisfaisants, performe mal sur les petit textes

**TODO : meilleur interpretation des resultats**
